# Script to generate a datachek with all the relevant information 
#### The information is stored per per subrun / run that is contained either in the datachecks and the MAGIC weather station

The data we will store in the datacheck will be the following:

#### <span style="color:darkred;">- Time:</span>
<span style="color:darkblue;">- Timestamp [datetime object] / Time elapsed [s]</span>
#### <span style="color:darkred;">- Pointing:</span>
<span style="color:darkblue;">- Azimuth [deg] / Zenith distance [deg]</span>
#### <span style="color:darkred;">- Intensity profiles:</span>
<span style="color:darkblue;">- Intensity at Half Peak Rate [p.e.] / Cosmics Rate at 422p.e. [ev / s / p.e.] / Delta Cosmics Rate at 422p.e. [ev / s / p.e.]</span>\
<span style="color:darkblue;">- Cosmics Spectral Index [] / Light Yield [p.e./p.e.]</span>
#### <span style="color:darkred;">- Weather:</span>
<span style="color:darkblue;">- Temperature [Cº] / Pressure [mmHg] / Humidity [%] / Wind Speed [km/h] / Wind Gust [km/h]</span>\
<span style="color:darkblue;">- Wind Speed Average [km/h] / TNG Dust [$\mu g/m^3$] / TNG Seeing [arcsecond] / Rain [tbd]</span>



## Datacheck `cosmics_intensity_spectrum` (subrun-wise)
Contains:\
yyyymmdd, ra_tel, dec_tel, cos_zenith, az_tel, runnumber,
       subrun, time, elapsed_time, corrected_elapsed_time,
       cosmics_rate, cosmics_cleaned_rate, intensity_at_half_peak_rate,
       ZD_corrected_intensity_at_half_peak_rate, cosmics_peak_rate,
       ZD_corrected_cosmics_peak_rate, cosmics_rate_at_422_pe,
       ZD_corrected_cosmics_rate_at_422_pe, cosmics_spectral_index,
       ZD_corrected_cosmics_spectral_index, intensity_spectrum_fit_p_value,
       intensity_at_reference_rate, diffuse_nsb_std,
       num_star_affected_pixels, anomalous_low_intensity_peak

## Datachek `runsummary` (run-wise)
Contains:\
runnumber, time, elapsed_time, min_altitude, mean_altitude,
       max_altitude, min_azimuth, max_azimuth, mean_azimuth, mean_ra,
       mean_dec, num_cosmics, num_pedestals, num_flatfield,
       num_unknown_ucts_trigger_tags, num_wrong_ucts_tags_in_cosmics,
       num_wrong_ucts_tags_in_pedestals, num_wrong_ucts_tags_in_flatfield,
       num_ucts_jumps, num_unknown_tib_trigger_tags,
       num_wrong_tib_tags_in_cosmics, num_wrong_tib_tags_in_pedestals,
       num_wrong_tib_tags_in_flatfield, num_pedestals_after_cleaning,
       num_contained_mu_rings, ff_charge_mean, ff_charge_mean_err,
       ff_charge_stddev, ff_time_mean, ff_time_mean_err,
       ff_time_stddev, ff_rel_time_stddev, ped_charge_mean,
       ped_charge_mean_err, ped_charge_stddev,
       ped_fraction_pulses_above10, ped_fraction_pulses_above30,
       cosmics_fraction_pulses_above10, cosmics_fraction_pulses_above30,
       mu_effi_mean, mu_effi_stddev, mu_width_mean, mu_width_stddev,
       mu_hg_peak_sample_mean, mu_hg_peak_sample_stddev,
       mu_intensity_mean, mean_number_of_pixels_nearby_stars
       
## Weather Station data
Contains:\
sun_alt, sun_az, fBits, mjd, temperature, pressure,
       windDirection, humidity, windSpeedCurrent, windGust,
       windSpeedAverage, windDirectionAverage, tempSensor, tngDust,
       tngSeeing, rain, state, Any, Mes, DP, diff1, is_dup,
       temperatureR

#### Import needed packages and scripts


In [62]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from datetime import datetime
import pickle, json, sys, os, glob, subprocess
import pandas as pd
pd.set_option("display.max_columns", None)
from scipy.optimize import curve_fit
from scipy.stats import chi2

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

sys.path.insert(0, os.getcwd() + "/../scripts/")
import auxiliar as aux
import geometry as geom

### Paths to data and results

In [26]:
# Number of rows for each job
n_rows = 6000

# Root path of this script
root = os.getcwd() + "/"
# Objects directory
root_data = root + "../data/"
root_tmp = root + "tmp/"

# Created files -------------------
# Filename of the datacheck dictionary
fname_dcheck_raw = root_data + "datachecks/dict_datachecks_raw.pkl" 

# Filename of the total dictionary
fname_dcheck_srunwise = root_data + "datachecks/dict_datachecks_srunwise.pkl"
fname_table_srunwise = root_data + "datachecks/table_datachecks_srunwise.csv"
# Filename of the dictionary merged
fname_dcheck_merged = root_data + "datachecks/dict_datachecks_merged.pkl" 
# Filename of the relation between run and night
fname_ws_run_relation = root_data + "ws/ws_run_relation.txt"
# Filename of reduced WS table + date array
fname_ws_reduced = root_data + "ws/reduced_ws_plus_dates.h5"

# Sources ----------------------
# Directory of all the night-wise datachecks
# root_dchecks = "/fefs/aswg/workspace/abelardo.moralejo/data/datachecks/night_wise/DL1_datacheck_"
root_dl1_dchecks_old = "/fefs/aswg/data/real/DL1/datacheck_files/night_wise"
root_dl1_dchecks_new = "/fefs/onsite/data/lst-pipe/LSTN-01/DL1/datacheck_files/night_wise"

# Weather station file
file_ws_db = root_data + "ws/WS2003-23.h5"

# Flags ------------------------
# Flags for computing or not different parts
# Compute the datacheck dictionary
compute_datacheck_dict = False
compute_ws_indexes = False
add_ws_data = False

process_inline = True
overwrite = False

# Power law parameters ref
ref_p0 =  1.74 
ref_p1 = -2.23

In [3]:
# Create needed folders
for d in [root_data + "datachecks/", root_tmp + "output_slurm/", root + "plots/"]:
    os.makedirs(d, exist_ok=True)

### Extracting dates and parameters of all runs/subruns, creating raw dcheck dict

In [4]:
%%time
if compute_datacheck_dict:

    run_number   = [] # Run numbers
    srun_number  = [] # Subrun numbers
    timestamps   = [] # Timestamps of each subrun
    time_elapsed = [] # Elapsed time of each subrun
    mean_azimuth         = [] # Mean azimuth of each run
    mean_zenith_distance = [] # Mean zenith of each run
    mean_ra  = [] # Mean right ascension
    mean_dec = [] # Mean dec
    cosmics_peak_rate              = [] # cosmics peak rate
    cosmics_rate_at_422_pe         = [] # cosmics rate at 422 pe
    intensity_at_half_peak_rate    = [] # intensity at half peak rate
    delta_cosmics_rate_at_422_pe   = [] # delta cosmics rate at 422 pe
    cosmics_spectral_index         = [] # cosmics_spectral_index
    delta_cosmics_spectral_index   = [] # delta cosmics spectral index
    zd_corrected_cosmics_peak_rate            = [] # ZD corrected cosmics peak rate   
    zd_corrected_cosmics_rate_at_422_pe       = [] # ZD corrected cosmics rate at 422 pe
    zd_corrected_intensity_at_half_peak_rate  = [] # ZD corrected intensity at half peak rate
    zd_corrected_delta_cosmics_rate_at_422_pe = [] # ZD corrected delta cosmics rate at 422 pe
    zd_corrected_cosmics_spectral_index       = [] # ZD corrected cosmics spectral index
    intensity_spectrum_fit_p_value = [] # intensity spectrum fit p value
    diffuse_nsb_std                = [] # diffuse nsb std
    charge_mean                    = [] # charge_mean
    light_yield                    = [] # Light yield

    # All the datachecks for all the nights
    dchecks = np.sort(glob.glob(root_dchecks + "*.h5"))

    # We iterate over all the datachecks
    for i, dcheck in enumerate(dchecks):

        print(f"Analysing... {i:3}/{len(dchecks)}") if i % 30 == 0 else None

        # The datacheck file of the run summary (runwise)
        ds = pd.read_hdf(dcheck, key="runsummary")
        # The datacheck file of the intensity spectrums (subrunwise)
        di = pd.read_hdf(dcheck, key="cosmics_intensity_spectrum")
        
        try:
            dp = pd.read_hdf(dcheck, key="pedestals")
            dp_flag = True
        except:
            dp_flag = False
            
        # Iterating over all the entries of each night, the subruns
        for j in range(len(ds)):

            # Reference run number
            runref = ds["runnumber"].iloc[j]
            
            # Intensity datacheck for only the subruns of the reference run
            di_run = di.query(f"runnumber == {runref}")            
            if dp_flag != False:
                dp_run = dp.query(f"runnumber == {runref}")
            
            # Subrun iteration and storing all the data we are interested in
            for k in range(len(di_run)):

                run_number.append(runref)
                srun_number.append(di_run["subrun"].iloc[k])
                timestamps.append(datetime.fromtimestamp(di_run["time"].iloc[k]))
                time_elapsed.append(di_run["corrected_elapsed_time"].iloc[k])
                
                mean_azimuth.append(ds["mean_azimuth"].iloc[j])
                mean_zenith_distance.append(np.arccos(di_run["cos_zenith"].iloc[k]))                
                mean_ra.append(ds["mean_ra"].iloc[j])
                mean_dec.append(ds["mean_dec"].iloc[j])
                
                cosmics_peak_rate.append(di_run["cosmics_peak_rate"].iloc[k])
                cosmics_rate_at_422_pe.append(di_run["cosmics_rate_at_422_pe"].iloc[k])
                intensity_at_half_peak_rate.append(di_run["intensity_at_half_peak_rate"].iloc[k])
                delta_cosmics_rate_at_422_pe.append(di_run["delta_cosmics_rate_at_422_pe"].iloc[k])
                cosmics_spectral_index.append(di_run["cosmics_spectral_index"].iloc[k])
                delta_cosmics_spectral_index.append(di_run["delta_cosmics_spectral_index"].iloc[k])
                
                zd_corrected_cosmics_peak_rate.append(
                    di_run["ZD_corrected_cosmics_peak_rate"].iloc[k])
                zd_corrected_cosmics_rate_at_422_pe.append(
                    di_run["ZD_corrected_cosmics_rate_at_422_pe"].iloc[k])
                zd_corrected_intensity_at_half_peak_rate.append(
                    di_run["ZD_corrected_intensity_at_half_peak_rate"].iloc[k])
                zd_corrected_delta_cosmics_rate_at_422_pe.append(
                    di_run["ZD_corrected_delta_cosmics_rate_at_422_pe"].iloc[k])
                zd_corrected_cosmics_spectral_index.append(
                    di_run["ZD_corrected_cosmics_spectral_index"].iloc[k])
                
                intensity_spectrum_fit_p_value.append(di_run["intensity_spectrum_fit_p_value"].iloc[k])
                diffuse_nsb_std.append(di_run["diffuse_nsb_std"].iloc[k])
                light_yield.append(di_run["light_yield"].iloc[k])
                
                if dp_flag != False:
                    try:
                        charge_mean.append(dp_run["charge_mean"].iloc[k])
                    except:
                        charge_mean.append(np.nan)
                else:
                    charge_mean.append(np.nan)
                    
    print(f"Analysing... {len(dchecks):3}/{len(dchecks)}\n")

    # Now we are going to sort looking to the timestamps
    _, run_number   = aux.sort_based(run_number, timestamps)
    _, srun_number  = aux.sort_based(srun_number, timestamps)
    _, time_elapsed = aux.sort_based(time_elapsed, timestamps)
    
    _, mean_azimuth         = aux.sort_based(mean_azimuth, timestamps)
    _, mean_zenith_distance = aux.sort_based(mean_zenith_distance, timestamps)
    _, mean_ra              = aux.sort_based(mean_ra, timestamps)
    _, mean_dec             = aux.sort_based(mean_dec, timestamps)

    _, cosmics_peak_rate            = aux.sort_based(cosmics_peak_rate, timestamps)
    _, cosmics_rate_at_422_pe       = aux.sort_based(cosmics_rate_at_422_pe, timestamps)
    _, intensity_at_half_peak_rate  = aux.sort_based(intensity_at_half_peak_rate, timestamps)
    _, delta_cosmics_rate_at_422_pe = aux.sort_based(delta_cosmics_rate_at_422_pe, timestamps)
    _, cosmics_spectral_index       = aux.sort_based(cosmics_spectral_index, timestamps)
    _, delta_cosmics_spectral_index = aux.sort_based(delta_cosmics_spectral_index, timestamps)
    
    _, zd_corrected_cosmics_peak_rate            = aux.sort_based(
        zd_corrected_cosmics_peak_rate, timestamps)
    _, zd_corrected_cosmics_rate_at_422_pe       = aux.sort_based(
        zd_corrected_cosmics_rate_at_422_pe, timestamps)
    _, zd_corrected_intensity_at_half_peak_rate  = aux.sort_based(
        zd_corrected_intensity_at_half_peak_rate, timestamps)
    _, zd_corrected_delta_cosmics_rate_at_422_pe = aux.sort_based(
        zd_corrected_delta_cosmics_rate_at_422_pe, timestamps)
    _, zd_corrected_cosmics_spectral_index       = aux.sort_based(
        zd_corrected_cosmics_spectral_index, timestamps)
    
    _, intensity_spectrum_fit_p_value = aux.sort_based(intensity_spectrum_fit_p_value, timestamps)
    _, diffuse_nsb_std                = aux.sort_based(diffuse_nsb_std, timestamps)
    _, charge_mean                    = aux.sort_based(charge_mean, timestamps)
    
    timestamps, light_yield = aux.sort_based(light_yield, timestamps)

    # Creating the data dictionary
    dict_dcheck_raw = {
        "run"      : np.array(run_number),
        "srun"     : np.array(srun_number),
        "time"     : np.array(timestamps),
        "telapsed" : np.array(time_elapsed),
        "az"  : np.rad2deg(mean_azimuth),
        "zd"  : np.rad2deg(mean_zenith_distance),
        "ra"  : np.array(mean_ra),
        "dec" : np.array(mean_dec),
        "cosmics_peak_rate"            : np.array(cosmics_peak_rate),
        "cosmics_rate_at_422_pe"       : np.array(cosmics_rate_at_422_pe),
        "intensity_at_half_peak_rate"  : np.array(intensity_at_half_peak_rate),
        "delta_cosmics_rate_at_422_pe" : np.array(delta_cosmics_rate_at_422_pe),
        "cosmics_spectral_index"       : np.array(cosmics_spectral_index),
        "delta_cosmics_spectral_index" : np.array(delta_cosmics_spectral_index),
        "ZD_corrected_cosmics_peak_rate"            : np.array(zd_corrected_cosmics_peak_rate),
        "ZD_corrected_cosmics_rate_at_422_pe"       : np.array(zd_corrected_cosmics_rate_at_422_pe),
        "ZD_corrected_intensity_at_half_peak_rate"  : np.array(zd_corrected_intensity_at_half_peak_rate),
        "ZD_corrected_delta_cosmics_rate_at_422_pe" : np.array(zd_corrected_delta_cosmics_rate_at_422_pe),
        "ZD_corrected_cosmics_spectral_index"       : np.array(zd_corrected_cosmics_spectral_index),
        "intensity_spectrum_fit_p_value" : np.array(intensity_spectrum_fit_p_value),
        "diffuse_nsb_std"                : np.array(diffuse_nsb_std),
        "charge_mean"                    : np.array(charge_mean),
        "light_yield"                    : np.array(light_yield),
    }        

    # Saving the objects in the objects directory
    with open(fname_dcheck_raw, "wb") as f:
        pickle.dump(dict_dcheck_raw, f, pickle.HIGHEST_PROTOCOL)  
else:
    # To read the file:
    with open(fname_dcheck_raw, "rb") as f:
            dict_dcheck_raw = pickle.load(f)    

CPU times: user 232 ms, sys: 148 ms, total: 380 ms
Wall time: 378 ms


## Creating a run-srun wise dictionary

In [5]:
%%time
dict_dcheck_srunwise = {}

# We create an entry per run
for run in np.unique(dict_dcheck_raw["run"]):
    dict_dcheck_srunwise[run] = {}

# Converting dcheck dictionary to total run and subrun dictionary
for i in range(len(dict_dcheck_raw["run"])):

    dict_dcheck_srunwise[dict_dcheck_raw["run"][i]][dict_dcheck_raw["srun"][i]] = {
        "time" :     dict_dcheck_raw["time"][i],
        "telapsed" : dict_dcheck_raw["telapsed"][i],
        
        "az"  : dict_dcheck_raw["az"][i],
        "zd"  : dict_dcheck_raw["zd"][i],
        "ra"  : dict_dcheck_raw["ra"][i],
        "dec" : dict_dcheck_raw["dec"][i],
        
        "cosmics_peak_rate" :            dict_dcheck_raw["cosmics_peak_rate"][i],
        "cosmics_rate_at_422_pe" :       dict_dcheck_raw["cosmics_rate_at_422_pe"][i],
        "intensity_at_half_peak_rate" :  dict_dcheck_raw["intensity_at_half_peak_rate"][i],
        "delta_cosmics_rate_at_422_pe" : dict_dcheck_raw["delta_cosmics_rate_at_422_pe"][i],
        "cosmics_spectral_index" :       dict_dcheck_raw["cosmics_spectral_index"][i],
        "delta_cosmics_spectral_index" : dict_dcheck_raw["delta_cosmics_spectral_index"][i],
        
        "ZD_corrected_cosmics_peak_rate" : 
            dict_dcheck_raw["ZD_corrected_cosmics_peak_rate"][i],
        "ZD_corrected_cosmics_rate_at_422_pe" : 
            dict_dcheck_raw["ZD_corrected_cosmics_rate_at_422_pe"][i],
        "ZD_corrected_intensity_at_half_peak_rate" : 
            dict_dcheck_raw["ZD_corrected_intensity_at_half_peak_rate"][i],
        "ZD_corrected_delta_cosmics_rate_at_422_pe" : 
            dict_dcheck_raw["ZD_corrected_delta_cosmics_rate_at_422_pe"][i],
        "ZD_corrected_cosmics_spectral_index" : 
            dict_dcheck_raw["ZD_corrected_cosmics_spectral_index"][i],
        
        "intensity_spectrum_fit_p_value" : dict_dcheck_raw["intensity_spectrum_fit_p_value"][i],
        "diffuse_nsb_std" :                dict_dcheck_raw["diffuse_nsb_std"][i],
        "charge_mean" :                    dict_dcheck_raw["charge_mean"][i],
        "light_yield" :                    dict_dcheck_raw["light_yield"][i]
    }

CPU times: user 3.98 s, sys: 411 ms, total: 4.39 s
Wall time: 4.39 s


## Merged dictionary

In [6]:
%%time
if compute_datacheck_dict:
    dict_dcheck_merged = {}
    for key in dict_dcheck_srunwise[list(dict_dcheck_srunwise.keys())[0]][0].keys():
        dict_dcheck_merged[key] = []

    for i, run in enumerate(dict_dcheck_srunwise.keys()):
        print(f"Appending... {i:3}/{len(dict_dcheck_srunwise.keys())}") if i % 500 == 0 else None

        for srun in dict_dcheck_srunwise[run].keys():
            for key in dict_dcheck_merged.keys():
                dict_dcheck_merged[key].append(dict_dcheck_srunwise[run][srun][key])

    # Saving the objects in the objects directory
    with open(fname_dcheck_merged, "wb") as f:
        pickle.dump(dict_dcheck_merged, f, pickle.HIGHEST_PROTOCOL)  
else:
    # To read the file:
    with open(fname_dcheck_merged, "rb") as f:
            dict_dcheck_merged = pickle.load(f)    

CPU times: user 4.96 s, sys: 1.52 s, total: 6.48 s
Wall time: 6.48 s


#### Reading the WS table and we reduce it to the part we are interested in

In [7]:
%%time
if compute_datacheck_dict:
    # Loading the weather station database
    df_ws = pd.read_hdf(file_ws_db)

    # Loading the timestamp of each entry in the datacheck dictionary
    dates_dcheck = dict_dcheck_raw["time"]

    # Getting the min and max dates
    maxdate, mindate = np.max(dates_dcheck), np.min(dates_dcheck)

    # Converting the weather station dates to datetime objects
    dates_ws = np.array([datetime.fromisoformat(str(d).split(".")[0]) for d in df_ws.index])

    # Getting the max date of the weather station
    maxdate_ws = np.max(dates_ws)

    # Masking the weather station data to the min and max dates of the datacheck dictionary
    mask_dates  = ((dates_ws > mindate) & (dates_ws < maxdate))
    # Masking also for day data, i.e. sun_alt > 0 (we set < -5 to have some error margin)
    mask_night = (df_ws["sun_alt"] < -5)

    total_mask = (mask_dates & mask_night)

    dates_ws = dates_ws[total_mask]
    df_ws    = df_ws[total_mask]

    with open(fname_ws_reduced, "wb") as f:
        pickle.dump([df_ws, dates_ws], f, pickle.HIGHEST_PROTOCOL)

else:
    # To read the file:
    with open(fname_ws_reduced, "rb") as f:
            df_ws, dates_ws = pickle.load(f)    

CPU times: user 185 ms, sys: 169 ms, total: 355 ms
Wall time: 350 ms


### Separating in bunchs of small number of jobs and writting into a txt file for slurm submissions

In [8]:
start_indexes, end_indexes, max_indexes = [], [], []

i, total = 0, 0
while total < len(dict_dcheck_raw["run"]):
    i1 = total
    i2 = total + n_rows - 1
    
    start_indexes.append(i1); end_indexes.append(i2)
    max_indexes.append(i1)
    
    i     += 1
    total += n_rows

print(f"With groups of {n_rows} subruns, the number of prepared jobs is {len(start_indexes)}")

With groups of 6000 subruns, the number of prepared jobs is 129


### Launching the jobs to the queue

In [9]:
%%time
INDEX = 115
if compute_datacheck_dict:
    for i, i1, i2 in zip(range(len(start_indexes[INDEX:])), start_indexes[INDEX:], end_indexes[INDEX:]):
        print(f"Computing... {i}/{len(start_indexes[INDEX:])}") if process_inline else None
        
        # Then we call the dl1_to_dl2 function in the dataprocessing_script.py
        str_args  = f"{i1} {i2} {fname_dcheck_raw} "
        str_args += f"{file_ws_db} {fname_ws_run_relation} {fname_ws_reduced}"
        python_command = f"python {root}script_datachecks.py write_ws_run_relation {str_args}"

        str_output = f"-o ./tmp/slurm_output/ws_run_relation_{i1}_{i2}.out"
        slurm_command = f"sbatch -p short -J ws_run_relation_{i1}_{i2} {str_output} --wrap='{python_command}'"

        command = python_command if process_inline else slurm_command
        subprocess.run(command, shell=True, text=True)

CPU times: user 13 μs, sys: 0 ns, total: 13 μs
Wall time: 23.4 μs


In [10]:
# Some slurm commands
!squeue -u juan.jimenez
# !scancel -u juan.jimenez 
# !seff 37423162

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON) 


### <span style="color:red;">------------------------------------------------------------------------------------------------------------------</span>
### <span style="color:red;">------------------------------------------------------------------------------------------------------------------</span>

### <span style="color:red;"> Wait untill the jobs are processed and then the results need to be fully stored </span>
### <span style="color:red;">------------------------------------------------------------------------------------------------------------------</span>
### <span style="color:red;">------------------------------------------------------------------------------------------------------------------</span>

#### Cleaning repeated lines in case the process was not done in a "clean" way

In [11]:
# def remove_duplicates(input_file, output_file):
# Read the file
with open(fname_ws_run_relation, "r") as file:
    lines = file.readlines()
unique_lines, counts = np.unique(lines, return_counts=True)

with open(fname_ws_run_relation, "w") as file:
    file.writelines(unique_lines)

print(f"Number of lines {len(lines)}, ratio = {len(lines) / n_rows:.2f}")
print(f"Number of repeated lines: {np.sum(counts > 1)}")

Number of lines 772398, ratio = 128.73
Number of repeated lines: 0


#### Now we read the results file where we associate each subrun to a entry of the WS data

In [12]:
%%time
# Reading the results
file_results_lines = np.loadtxt(fname_ws_run_relation, dtype=str, delimiter=",")

# Creating a dictionary to organise them
dict_results_ws_run_relation = {}

for line in file_results_lines:
    runsubrun, date_str = line
    run, srun = runsubrun.split("-")
    run, srun = int(run), int(srun)

    date_str = date_str if date_str != "None" else None

    try:
        dict_results_ws_run_relation[run][srun] = date_str    
    except KeyError:
        dict_results_ws_run_relation[run] = {srun : date_str}
    

CPU times: user 2.47 s, sys: 104 ms, total: 2.58 s
Wall time: 2.57 s


#### Now the weather station data can be added to the total dictionary

In [13]:
%%time
if add_ws_data:
    for i, run in enumerate(dict_dcheck_srunwise.keys()):
        print(f"Adding data... {i:6}/{len(dict_dcheck_srunwise.keys())} runs") if i % 500 == 0 else None

        for srun in dict_dcheck_srunwise[run].keys():

            try:
                string_date = dict_results_ws_run_relation[run][srun]
                empty_flag = False if string_date != None else True

            except KeyError:
                empty_flag = True

            if not empty_flag:
                try:
                    dict_dcheck_srunwise[run][srun]["weather"] = {
                        "temperature" :        df_ws.loc[string_date]["temperature"],      # degree celsius
                        "pressure" :           df_ws.loc[string_date]["pressure"],         # mmHg
                        "humidity" :           df_ws.loc[string_date]["humidity"],         # %
                        "wind_speed" :         df_ws.loc[string_date]["windSpeedCurrent"], # km/h
                        "wind_gust" :          df_ws.loc[string_date]["windGust"],         # km/h
                        "wind_speed_average" : df_ws.loc[string_date]["windSpeedAverage"], # km/s
                        "tng_dust" :           df_ws.loc[string_date]["tngDust"],          # ug/m3
                        "tng_seeing" :         df_ws.loc[string_date]["tngSeeing"],        # arcseconds
                        "rain" :               df_ws.loc[string_date]["rain"],             #
                    }
                except KeyError:
                    print(f"KeyError in Run {run}, Subrun {srun} with entry ID {string_date}.")
                    dict_dcheck_srunwise[run][srun]["weather"] = {
                        "temperature" :        np.nan, # degree celsius
                        "pressure" :           np.nan, # mmHg
                        "humidity" :           np.nan, # %
                        "wind_speed" :         np.nan, # km/h
                        "wind_gust" :          np.nan, # km/h
                        "wind_speed_average" : np.nan, # km/s
                        "tng_dust" :           np.nan, # ug/m3
                        "tng_seeing" :         np.nan, # arcseconds
                        "rain" :               np.nan, #
                    } 

            else:            
                dict_dcheck_srunwise[run][srun]["weather"] = {
                    "temperature" :        np.nan, # degree celsius
                    "pressure" :           np.nan, # mmHg
                    "humidity" :           np.nan, # %
                    "wind_speed" :         np.nan, # km/h
                    "wind_gust" :          np.nan, # km/h
                    "wind_speed_average" : np.nan, # km/s
                    "tng_dust" :           np.nan, # ug/m3
                    "tng_seeing" :         np.nan, # arcseconds
                    "rain" :               np.nan, #
                }
    # Saving the object
    with open(fname_dcheck_srunwise, "wb") as f:
        pickle.dump(dict_dcheck_srunwise, f, pickle.HIGHEST_PROTOCOL)
else:
    # To read the file:
    with open(fname_dcheck_srunwise, "rb") as f:
            dict_dcheck_srunwise = pickle.load(f) 

CPU times: user 16.2 s, sys: 2.75 s, total: 19 s
Wall time: 19 s


### Create a RunWise CSV file where we will add the computed runwise information for drdi

In [65]:
columns = [
    "obs_id", "n_subruns", "timestamp", "telapsed", "az", "zd", "ra", "dec",
    
    "cosmics_peak_rate", "stdv_cosmics_peak_rate", 
    "cosmics_rate_at_422_pe", "stdv_cosmics_rate_at_422_pe", 
    "intensity_at_half_peak_rate", "stdv_intensity_at_half_peak_rate",
    "delta_cosmics_rate_at_422_pe", "stdv_delta_cosmics_rate_at_422_pe",
    "cosmics_spectral_index", "stdv_cosmics_spectral_index",
    "delta_cosmics_spectral_index", "stdv_delta_cosmics_spectral_index",
    "ZD_corrected_cosmics_peak_rate", "stdv_ZD_corrected_cosmics_peak_rate", 
    "ZD_corrected_cosmics_rate_at_422_pe", "stdv_ZD_corrected_cosmics_rate_at_422_pe",
    "ZD_corrected_intensity_at_half_peak_rate", "stdv_ZD_corrected_intensity_at_half_peak_rate", 
    "ZD_corrected_delta_cosmics_rate_at_422_pe", "stdv_ZD_corrected_delta_cosmics_rate_at_422_pe",
    "ZD_corrected_cosmics_spectral_index", "stdv_ZD_corrected_cosmics_spectral_index",
    
    "intensity_spectrum_fit_p_value", "diffuse_nsb_std", 
    "charge_mean", "stdv_charge_mean", "light_yield",
    
    "run_drdi_fit_p0", "run_drdi_fit_p1", "run_drdi_fit_u_p0", "run_drdi_fit_u_p1",
    "run_drdi_fit_chi2", "run_drdi_fit_pvalue", "run_drdi_fit_ndf", "run_drdi_fit_error_flag",
    
    "temperature", "pressure", "humidity", "wind_speed", "wind_gust",
    "wind_speed_average", "tng_dust", "tng_seeing", "rain",
]

df = pd.DataFrame(columns=columns)

for i, run in enumerate(dict_dcheck_srunwise.keys()):
    print(f"Adding data... {i:6}/{len(dict_dcheck_srunwise.keys())} runs") if i % 500 == 0 else None
    
    # ----------------------------#
    # Run-wise drdi fit with a line
    #-----------------------------#
    
    d = dict_dcheck_srunwise[run]
    
    _sruns    = np.array(list(d.keys()))
    _time     = np.array([d[s]["time"] for s in _sruns])
    _telapsed = np.array([d[s]["telapsed"] for s in _sruns])
    _drdi     = np.array([d[s]["ZD_corrected_cosmics_rate_at_422_pe"] for s in _sruns])
    _u_drdi   = np.array([d[s]["ZD_corrected_delta_cosmics_rate_at_422_pe"] for s in _sruns])
    _mean_drdi, _std_drdi = np.mean(_drdi), np.std(_drdi)

    # Excluding NaN values
    mask_nan = (np.isnan(_drdi) | np.isnan(_u_drdi))
    # We will exclude too small error that will lead to wrong fit
    mask_zero_error = np.array([e < 1e-5 for e in _u_drdi])
    # Also excluding very high or low points (most probably due to car flashes or clouds)
    mask_high_drdi  = np.array([d > 2.4 or d > _mean_drdi + 2 * _std_drdi for d in _drdi])
    mask_low_drdi  = np.array([d < _mean_drdi - 2 * _std_drdi for d in _drdi])
    # Total mask
    mask_fit = ~(mask_nan | mask_zero_error | mask_high_drdi | mask_low_drdi)
        
    x_fit = np.cumsum(_telapsed)[mask_fit]
    y_fit = _drdi[mask_fit]
    yerr_fit = _u_drdi[mask_fit]
    
    if len(x_fit) > 2:
        params, pcov, info, _, _ = curve_fit(
            f     = geom.straight_line,
            xdata = x_fit,
            ydata = y_fit,
            sigma = yerr_fit,
            p0    = [0, 1],
            full_output = True,
        )

        fit_p0, fit_p1  = params
        fit_delta_p0 = np.sqrt(pcov[0, 0])
        fit_delta_p1 = np.sqrt(pcov[1, 1])
        fit_ndf = len(x_fit)
        fit_chi2 = np.sum(info["fvec"] ** 2)
        fit_pvalue   = 1 - chi2.cdf(fit_chi2, fit_ndf)    
        fit_error_flag = False
    else:
        fit_p0, fit_p1  = np.nan, np.nan
        fit_delta_p0, fit_delta_p1 = np.nan, np.nan
        fit_ndf = len(x_fit)
        fit_chi2, fit_pvalue = np.nan, np.nan
        fit_error_flag = True     
    
    row = [
        run,
        len(dict_dcheck_srunwise[run].keys()), # Nsubruns
        dict_dcheck_srunwise[run][list(dict_dcheck_srunwise[run].keys())[0]]["time"], # timestamp
        np.sum([dict_dcheck_srunwise[run][s]["telapsed"] 
                for s in dict_dcheck_srunwise[run].keys()]), # telapsed
        np.mean([dict_dcheck_srunwise[run][s]["az"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # az
        np.mean([dict_dcheck_srunwise[run][s]["zd"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # zd
        np.mean([dict_dcheck_srunwise[run][s]["ra"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # ra
        np.mean([dict_dcheck_srunwise[run][s]["dec"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # dec
        
        np.mean([dict_dcheck_srunwise[run][s]["cosmics_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean cosmics_peak_rate
        np.std([dict_dcheck_srunwise[run][s]["cosmics_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv cosmics_peak_rate
        np.mean([dict_dcheck_srunwise[run][s]["cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean cosmics_rate_at_422_pe
        np.std([dict_dcheck_srunwise[run][s]["cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv cosmics_rate_at_422_pe
        np.mean([dict_dcheck_srunwise[run][s]["intensity_at_half_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean intensity_at_half_peak_rate
        np.std([dict_dcheck_srunwise[run][s]["intensity_at_half_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv intensity_at_half_peak_rate
        np.mean([dict_dcheck_srunwise[run][s]["delta_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean delta_cosmics_rate_at_422_pe
        np.std([dict_dcheck_srunwise[run][s]["delta_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv delta_cosmics_rate_at_422_pe
        np.mean([dict_dcheck_srunwise[run][s]["cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean cosmics_spectral_index
        np.std([dict_dcheck_srunwise[run][s]["cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv cosmics_spectral_index
        np.mean([dict_dcheck_srunwise[run][s]["delta_cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean delta_cosmics_spectral_index
        np.std([dict_dcheck_srunwise[run][s]["delta_cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv delta_cosmics_spectral_index
        np.mean([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean ZD_corrected_cosmics_peak_rate
        np.std([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv ZD_corrected_cosmics_peak_rate
        np.mean([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean ZD_corrected_cosmics_rate_at_422_pe
        np.std([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv ZD_corrected_cosmics_rate_at_422_pe
        np.mean([dict_dcheck_srunwise[run][s]["ZD_corrected_intensity_at_half_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean ZD_corrected_intensity_at_half_peak_rate
        np.std([dict_dcheck_srunwise[run][s]["ZD_corrected_intensity_at_half_peak_rate"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv ZD_corrected_intensity_at_half_peak_rate
        np.mean([dict_dcheck_srunwise[run][s]["ZD_corrected_delta_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean ZD_corrected_delta_cosmics_rate_at_422_pe
        np.std([dict_dcheck_srunwise[run][s]["ZD_corrected_delta_cosmics_rate_at_422_pe"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv ZD_corrected_delta_cosmics_rate_at_422_pe
        np.mean([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # mean ZD_corrected_cosmics_spectral_index
        np.std([dict_dcheck_srunwise[run][s]["ZD_corrected_cosmics_spectral_index"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv ZD_corrected_cosmics_spectral_index
        
        np.mean([dict_dcheck_srunwise[run][s]["intensity_spectrum_fit_p_value"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # intensity_spectrum_fit_p_value
        np.mean([dict_dcheck_srunwise[run][s]["diffuse_nsb_std"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # diffuse_nsb_std
        np.mean([dict_dcheck_srunwise[run][s]["charge_mean"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # charge_mean
        np.std([dict_dcheck_srunwise[run][s]["charge_mean"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # stdv charge_mean
        np.mean([dict_dcheck_srunwise[run][s]["light_yield"] 
                 for s in dict_dcheck_srunwise[run].keys()]), # light_yield
        
        fit_p0, # fit p0
        fit_p1, # fit p1
        fit_delta_p0, # fit u_p0
        fit_delta_p1, # fit u_p1
        fit_chi2, # fit chi2
        fit_pvalue, # fit pvalue
        fit_ndf, # fit ndf
        fit_error_flag, # fit error flag
        
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["temperature"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # temperature
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["pressure"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # pressure
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["humidity"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # humidity
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["wind_speed"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # wind_speed
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["wind_gust"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # wind_gust
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["wind_speed_average"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # wind_speed_average
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["tng_dust"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # tng_dust
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["tng_seeing"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # tng_seeing
        np.nanmean([dict_dcheck_srunwise[run][s]["weather"]["rain"] 
                    for s in dict_dcheck_srunwise[run].keys()]), # rain   
    ]
    
    df.loc[i] = row

display(df)

# Saving the table as csv
df.to_csv(fname_table_srunwise)

Adding data...      0/7771 runs
Adding data...    500/7771 runs
Adding data...   1000/7771 runs
Adding data...   1500/7771 runs
Adding data...   2000/7771 runs
Adding data...   2500/7771 runs
Adding data...   3000/7771 runs
Adding data...   3500/7771 runs
Adding data...   4000/7771 runs
Adding data...   4500/7771 runs
Adding data...   5000/7771 runs
Adding data...   5500/7771 runs
Adding data...   6000/7771 runs
Adding data...   6500/7771 runs
Adding data...   7000/7771 runs
Adding data...   7500/7771 runs


,obs_id,n_subruns,timestamp,telapsed,az,zd,ra,dec,cosmics_peak_rate,stdv_cosmics_peak_rate,cosmics_rate_at_422_pe,stdv_cosmics_rate_at_422_pe,intensity_at_half_peak_rate,stdv_intensity_at_half_peak_rate,delta_cosmics_rate_at_422_pe,stdv_delta_cosmics_rate_at_422_pe,cosmics_spectral_index,stdv_cosmics_spectral_index,delta_cosmics_spectral_index,stdv_delta_cosmics_spectral_index,ZD_corrected_cosmics_peak_rate,stdv_ZD_corrected_cosmics_peak_rate,ZD_corrected_cosmics_rate_at_422_pe,stdv_ZD_corrected_cosmics_rate_at_422_pe,ZD_corrected_intensity_at_half_peak_rate,stdv_ZD_corrected_intensity_at_half_peak_rate,ZD_corrected_delta_cosmics_rate_at_422_pe,stdv_ZD_corrected_delta_cosmics_rate_at_422_pe,ZD_corrected_cosmics_spectral_index,stdv_ZD_corrected_cosmics_spectral_index,intensity_spectrum_fit_p_value,diffuse_nsb_std,charge_mean,stdv_charge_mean,light_yield,run_drdi_fit_p0,run_drdi_fit_p1,run_drdi_fit_u_p0,run_drdi_fit_u_p1,run_drdi_fit_chi2,run_drdi_fit_pvalue,run_drdi_fit_ndf,run_drdi_fit_error_flag,temperature,pressure,humidity,wind_speed,wind_gust,wind_speed_average,tng_dust,tng_seeing,rain
0,1615,61,2019-11-23 23:40:47.731307,1479.185384,265.754848,31.359290,8.676201,22.059712,7.252669,0.108430,1.879985,0.029587,127.976556,1.207705,0.018524,0.007872,-2.179461,0.052953,0.059882,0.025505,9.615377,0.207266,2.045992,0.024084,124.759916,1.190506,0.020157,0.008575,-2.126104,0.053143,0.426504,1.465696,1.843949,0.015313,1.155216,2.051726,-0.000015,0.007395,0.000008,167.713018,7.749357e-13,57,False,7.588689,787.011475,45.536721,14.823115,18.493115,14.341311,0.380000,1.710000,0.0
1,1616,62,2019-11-24 00:11:26.987927,1492.465697,95.332874,30.722616,83.659785,21.844168,7.338338,0.111450,1.923990,0.028007,127.591533,1.268291,0.019916,0.007579,-2.165521,0.048141,0.062913,0.023863,9.644406,0.193980,2.085128,0.018315,124.538413,1.282795,0.021588,0.008210,-2.114527,0.046639,0.383385,1.678245,2.113709,0.006173,1.176689,2.077890,0.000015,0.005083,0.000006,75.387183,1.017617e-01,61,False,7.951452,786.889839,46.629677,14.410645,18.621774,14.384194,0.277419,2.077581,0.0
2,1617,35,2019-11-24 00:45:42.507987,877.161720,273.215706,44.511343,8.675483,22.076830,6.785111,0.082079,1.655979,0.025860,127.771350,0.929714,0.015800,0.007686,-2.245861,0.073846,0.058021,0.028475,10.181520,0.112144,2.049730,0.018668,118.418470,1.154316,0.019581,0.009608,-2.127638,0.073560,0.495740,1.490406,1.873624,0.007748,1.157435,2.043657,0.000024,0.005334,0.000010,44.641273,8.496942e-02,33,False,7.710571,786.620000,51.568000,14.998286,19.397429,15.432286,0.260000,2.140000,0.0
3,1618,81,2019-11-24 01:07:29.246502,1822.499924,108.322078,18.014448,83.668157,21.922445,7.559337,0.107014,2.114495,0.213096,126.812227,1.208264,0.037531,0.064773,-2.067526,0.386137,0.094747,0.130588,8.332031,0.166631,2.164965,0.212443,125.807134,1.174829,0.038297,0.065847,-2.051108,0.385296,0.395566,1.690460,2.128744,0.011473,1.160424,2.105509,0.000009,0.005707,0.000007,310.499037,0.000000e+00,75,False,7.585802,786.379877,60.808272,14.600370,19.541605,13.873580,0.086790,2.463827,0.0
4,1619,30,2019-11-24 01:45:25.986127,922.360651,279.314177,57.571760,8.704526,22.086164,5.736650,0.153067,1.272401,0.041651,126.917053,1.125708,0.013388,0.006202,-2.304494,0.063341,0.063585,0.028160,8.413803,0.245827,2.045831,0.022481,112.330784,0.997127,0.021450,0.009606,-2.085522,0.060519,0.460160,1.544312,1.935010,0.005709,1.161393,2.052083,-0.000015,0.006106,0.000012,29.995632,3.634267e-01,28,False,7.658333,786.082000,58.738667,16.784667,21.891333,15.738000,0.030000,2.570000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7766,16751,267,2024-02-13 04:33:34.736869,2349.360992,210.116608,18.940156,188.120553,12.393466,43.504798,1.158610,1.569482,0.028584,37.849459,0.286231,0.026539,0.011168,-2.308690,0.104768,0.102670,0.043077,48.421273,0.991914,1.611333,0.